# Modelo Predictivo Precio Airbnb | Proyecto Final Ciencia de Datos 
---

## Objetive: 
To analyze Airbnb listings in order to predict listing prices and understand what factors have the biggest impact on said price. 

## Initial strategic decisions
1. *Raw price versus Log price*: The model will predict log prices instead of the raw prices on the listings in order to prioritize treating proportional differences in pricing the same across the board. This decision was made in order to gain a more symmetric, normal distribution so that the linear regression model works better. The tradeoff being the retransformation bias that occurs when converting from the log price back into dollar amounts.
2. *Models*: For this analysis there will only be one general model that predicts price listings regardless of room types. 
3. *Feature engineering*: In order to gain better insight into the factors affecting Airbnb prices there will be extensive feature engineering and the project will have a big focus on this area of the analysis. 

# 1.1 Exploratory Data Analysis
---
To begin the exploratory data analysis we will be answering the following questions:
- How many rows and columns does the dataset have?
- What does each column represent?

In [28]:
# import libraries 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib import style
from scipy.stats import pearsonr
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import seaborn as sns 
import csv
import datetime as dt

# import data 
excel = pd.read_excel('datosViviendas.xlsx', header = None)

# drop empty cells and convert lines to strings to read as a csv 
lines = excel[0].dropna().astype(str).tolist()
rows = list(csv.reader(lines))
header = rows[0]

# drop rows with incomplete or exceeding fields 
completeRows = [row for row in rows[1:] if len(row) == len(header)]

# Initial EDA 
df = pd.DataFrame(completeRows, columns = header)
variables = list(df.columns)

print(f"The dataset contains {df.shape[0]} rows and {df.shape[1]} columns.\n")
print(f"The dataset contains the following variables: {variables}.\n")
print(f"The first rows of the dataset are:\n{df.head().to_string()}\n")


The dataset contains 63943 rows and 29 columns.

The dataset contains the following variables: ['id', 'log_price', 'property_type', 'room_type', 'amenities', 'accommodates', 'bathrooms', 'bed_type', 'cancellation_policy', 'cleaning_fee', 'city', 'description', 'first_review', 'host_has_profile_pic', 'host_identity_verified', 'host_response_rate', 'host_since', 'instant_bookable', 'last_review', 'latitude', 'longitude', 'name', 'neighbourhood', 'number_of_reviews', 'review_scores_rating', 'thumbnail_url', 'zipcode', 'bedrooms', 'beds'].

The first rows of the dataset are:
         id           log_price property_type        room_type                                                                                                                                                                                                                                                                                                                amenities accommodates bathrooms  bed_type cancellation_

# 1.1 Structural Overview
---
With my beginning analysis I made the following observations about the dataset:

| Variables              | What is it?                                                             | Data Type   |
| ---------------------- | ----------------------------------------------------------------------- | ----------- |
| id                     | Number that identifies the listing.                                     | str         |
| log_price              | Logarithmic price of the listing.                                       | float       |
| property_type          | Type of property (e.g. House/Apartment).                                | categorical |
| room_type              | Type of room (e.g. )                                                    | categorical |
| amenities              | List of amenities (e.g. TV, Wireless Internet, Air Conditioning, etc.)  | FE          |
| accommodates           | Number of people the property accommodates                              | int         |
| bathrooms              | number of bathrooms                                                     | float       |
| bed_type               | Type of bed                                                             | categorical |
| cancellation_policy    | How strict the cancellation policy is (e.g. strict/ moderate/flexible). | categorical |
| cleaning_fee           | Whether or not a cleaning fee exists.                                   | boolean     |
| city                   | The city the property is located in.                                    | categorical |
| description            | A description of the property.                                          | FE          |
| first_review           | Date of the first review.                                               | FE          |
| host_has_profile_pic   | Whether or not the listing host has a pfp.                              | boolean     |
| host_identity_verified | Whether or not the hosts identity is verified.                          | boolean     |
| host_response_rate     | Percent of times host responds to messages.                             | float       |
| host_since             | Date the Airbnb user became host.                                       | FE          |
| instant_bookable       | Whether or not the property is instantly bookable.                      | boolean     |
| last_review            | The last review's date.                                                 | FE          |
| latitude               | Latitude value.                                                         | FE          |
| longitude              | Longitude value.                                                        | FE          |
| name                   | Property/listing name.                                                  | FE          |
| neighbourhood          | What neighbourhood the property is in.                                  | categorical |
| number_of_reviews      | The number of reviews the listing has.                                  | int         |
| review_scores_rating   | The average rating of the review scores.                                | float       |
| thumbnail_url          | Url for posts thumbnail.                                                | FE          |
| zipcode                | Property's zip code.                                                    | FE          |
| bedrooms               | Number of bedrooms the property has.                                    | int         |
| beds                   | Number of beds the property has.                                        | int         |

In [29]:
# Plan out what dtypes each column should be converted to for analysis and modeling
numeric_columns = ['accommodates', 'number_of_reviews']
float_columns = ['log_price', 'bathrooms', 'review_scores_rating', 'host_response_rate', 'bedrooms', 'beds']
categorical_columns = ['property_type', 'room_type', 'bed_type', 'cancellation_policy', 'city', 'neighbourhood']
boolean_columns = ['cleaning_fee', 'host_has_profile_pic', 'host_identity_verified', 'instant_bookable', ]
feature_engineering_columns = ['amenities', 'description', 'first_review', 'host_since', 'last_review', 'latitude', 'longitude', 'name', 'thumbnail_url', 'zipcode']

# Visualize unique values to determine if any erroneous values exist 
for col in numeric_columns:
    print(f"Unique values for {col}:\n{df[col].unique()}\n")

for col in float_columns:
    print(f"Unique values for {col}:\n{df[col].unique()}\n")

for col in categorical_columns:
    print(f"Unique values for {col}:\n{df[col].unique()}\n")

for col in boolean_columns:
    print(f"Unique values for {col}:\n{df[col].unique()}\n")

Unique values for accommodates:
<StringArray>
[ '3',  '7',  '5',  '4',  '2',  '6',  '8',  '1',  '9', '10', '16', '12', '11',
 '14', '13', '15']
Length: 16, dtype: str

Unique values for number_of_reviews:
<StringArray>
[  '2',   '6',  '10',   '0',   '4',   '3',  '15',   '9', '159',  '82',
 ...
 '243', '382', '380', '358', '265', '354', '376', '315', '1.0', '341']
Length: 356, dtype: str

Unique values for log_price:
<StringArray>
[ '5.010635294096256', '5.1298987149230735',  '4.976733742420574',
  '6.620073206530356',   '4.74493212836325',  '4.442651256490317',
 '4.4188406077965965',  '4.787491742782046',   '3.58351893845611',
  '4.605170185988092',
 ...
 '6.1463292576688975',  '6.817830571454152', '5.8971538676367405',
  '5.823045895483018', '6.1441856341256464', '7.0255383146385215',
   '7.01301578963963',  '6.714170529909472',  '6.045005314036013',
    '6.3578422665081']
Length: 737, dtype: str

Unique values for bathrooms:
<StringArray>
['1.0', '1.5', '2.0',    '', '2.5', '3.0', '0

Exploring each variable individually I observed some erroneous data that was in incorrect columns or in a bad format and will need to be dealt with.

| Variable               | Erroneous Data                                                                                                                                                                      |
| ---------------------- | ----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| property_type          | Not erroneous but strange categories like tipi, castle, earth house, cave, train, island, yurt, timeshare, lighthouse (maybe yurt is erroneous does not seem like a property type). |
| bathrooms              | Empty strings.                                                                                                                                                                      |
| host_has_profile_pic   | Empty strings, 'Wireless Internet""'                                                                                                                                                |
| host_identity_verified | Empty strings, 'Air conditioning""'                                                                                                                                                 |
| host_response_rate     | 'Kitchen', empty strings.                                                                                                                                                           |
| instant_bookable       | 'Family/kid friendly""'                                                                                                                                                             |
| neighbourhood          | so many different values its very hard to actually tell if any are erroneous like hell's kitchen.                                                                                   |
| number_of_reviews      | 1.0 (all other observed values were ints and it is illogical to have decimal reviews).                                                                                              |
| review_scores_rating   | Empty strings, 'Real Bed'.                                                                                                                                                          |
| bedrooms               | Empty strings, NYC.                                                                                                                                                                 |
| beds                   | Empty strings, a listing description.                                                                                                                                               |


In [ ]:
# Find index of possible misaligned row
misaligned_row_index = df[df['host_has_profile_pic'] == 'Wireless Internet""'].index
print(df.iloc[misaligned_row_index].to_string())

# See number of boolean values that are empty strings
for col in boolean_columns:
    print(f"Value counts for {col}:\n{df[col].value_counts()}\n")

Index of possible misaligned row: RangeIndex(start=63375, stop=63376, step=1)

            id           log_price property_type     room_type                                                                                                                                                                                                                                                                       amenities accommodates bathrooms  bed_type cancellation_policy cleaning_fee city                                                                                                                                                                                                                                                                                                                                                                                                                                        description first_review host_has_profile_pic host_identity_verified host_response_rate host

In [31]:
# Map boolean values to True or False and list any unwanted characters
boolean_mapping = {'t': True, 'f': False, 'True': True, 'False': False}
chars_to_replace = {'%': ''}

# Transform columns to their corresponding dtypes
df['accommodates'] = pd.to_numeric(df['accommodates'], errors="coerce").astype('float64')
df['number_of_reviews'] = pd.to_numeric(df['number_of_reviews'], errors="coerce").astype('float64')

df['log_price'] = pd.to_numeric(df['log_price'], errors="coerce").astype('float64')
df['bathrooms'] = pd.to_numeric(df['bathrooms'], errors="coerce").astype('float64')
df['review_scores_rating'] = pd.to_numeric(df['review_scores_rating'], errors="coerce").astype('float64')
df['host_response_rate'] = pd.to_numeric(df['host_response_rate'].replace(chars_to_replace), errors="coerce").astype('float64')
df['bedrooms'] = pd.to_numeric(df['bedrooms'], errors="coerce").astype('float64')
df['beds'] = pd.to_numeric(df['beds'], errors="coerce").astype('float64')

df['cleaning_fee'] = df['cleaning_fee'].replace(chars_to_replace).astype('bool')
df['host_has_profile_pic'] = df['host_has_profile_pic'].replace(chars_to_replace).astype('bool')
df['host_identity_verified'] = df['host_identity_verified'].replace(chars_to_replace).astype('bool')
df['instant_bookable'] = df['instant_bookable'].replace(chars_to_replace).astype('bool')

print(f"The dataset contains the following data types after transformation:\n{df.dtypes.to_string()}")

The dataset contains the following data types after transformation:
id                            str
log_price                 float64
property_type                 str
room_type                     str
amenities                     str
accommodates              float64
bathrooms                 float64
bed_type                      str
cancellation_policy           str
cleaning_fee                 bool
city                          str
description                   str
first_review                  str
host_has_profile_pic         bool
host_identity_verified       bool
host_response_rate        float64
host_since                    str
instant_bookable             bool
last_review                   str
latitude                      str
longitude                     str
name                          str
neighbourhood                 str
number_of_reviews         float64
review_scores_rating      float64
thumbnail_url                 str
zipcode                       str
bedrooms      